# Coupled Acoustic–Elastic Wave Forward Simulation

This tutorial was prepared by Jéssica Santos (jessicasales@usp.br)

This notebook demonstrates the coupled fluid–solid forward solver implemented in **spyro**, based on the $P$–$u_s$ formulation of [Li et al. (2019)](https://doi.org/10.1093/jge/gxy017). It covers the construction of the velocity model, configuration of the simulation parameters, execution of the forward solver, and saving and visualization of the results.

The example runs a **2D case with a sinusoidal non-planar fluid–solid interface** on a GMSH-generated mesh, using the sequential solution scheme.

**Prerequisites**: Basic understanding of the forward modeling tutorial.

If you are running this notebook in Google Colab, please copy the following code into a code block before running the notebook:

```python
# For use in Colab only:
try:
    import firedrake
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    import firedrake

!pip install git+https://github.com/NDF-Poli-USP/spyro.git
```

## 1. Imports

Standard libraries for file I/O, timing, and memory profiling, alongside **Firedrake** (FEM framework) and **spyro** (seismic wave solver).

In [14]:
import sys
sys.path.insert(0, '/workspaces/spyro2')
import os
import sys
import time
import resource
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.transforms import Affine2D
from IPython.display import Image, display

import firedrake as fire
from firedrake import Function, FunctionSpace, triplot, tripcolor

from spyro.solvers.acoustic_elastic_wave import AcousticElasticWave
from spyro.solvers.acoustic_solver_construction_no_pml import velocity_fluid
from spyro.utils.utils import communicate
from spyro.io.segy_io import create_segy_from_grid

SCRIPT_DIR = Path('.').resolve()

ModuleNotFoundError: No module named 'spyro.solvers.acoustic_elastic_wave'

## 2. Helper functions

### 2.1 SEG-Y velocity model builder

`build_segy` creates a 2D velocity model in SEG-Y format, required when using GMSH-generated meshes. The model assigns `v_fluid` above the interface and `v_solid` below it. The interface can be:
- a constant depth (float),
- a callable function $z = f(x)$ (e.g. sinusoidal),
- a path to a text file with $(x, z)$ pairs.

> **Note:** The example in this notebook uses a **sinusoidal non-planar interface** with a GMSH-generated mesh, so this function is called during the mesh setup step.

In [ ]:
def build_segy(filename, length_z, length_x, dz, interface, v_fluid, v_solid, dx=None):
    dx = dz if dx is None else dx
    z = np.linspace(0.0, -length_z, int(round(length_z / dz)) + 1)
    x = np.linspace(0.0, length_x, int(round(length_x / dx)) + 1)
    X, Z = np.meshgrid(x, z, indexing="ij")
    if callable(interface):
        z_int = np.broadcast_to(interface(x), x.shape)
    elif isinstance(interface, (str, Path)):
        d = np.loadtxt(interface)
        z_int = np.interp(x, d[:, 0], d[:, 1])
    else:
        z_int = np.full(x.shape, float(interface))
    field = lambda v: np.broadcast_to(v(Z, X), Z.shape) if callable(v) else np.full(Z.shape, float(v))
    v = np.where(Z > z_int[:, None], field(v_fluid), field(v_solid))
    create_segy_from_grid(v[:, ::-1].astype(np.float32), str(filename))

### 2.2 SPECFEM2D interface reader

`specfem_interface` reads the bathymetric interface from a SPECFEM2D-format `.dat` file and returns a function $z(x)$ that interpolates the interface depth at any horizontal position $x$. The `index` argument selects which interface to read from the file (0-indexed).

> **Note:** The example in this notebook uses a sinusoidal interface defined analytically, so this function is not called. It is included here for reference when running the large irregular case with SPECFEM2D bathymetry.

In [ ]:
def specfem_interface(filename, index, length_z):
    with open(filename) as f:
        rows = [line.split() for line in f
                if line.strip() and not line.lstrip().startswith("#")]
    it = iter(rows[1:])
    interfaces = []
    for _ in range(int(rows[0][0])):
        n = int(next(it)[0])
        interfaces.append(np.array([[float(a), float(b)] for a, b in (next(it) for _ in range(n))]))
    xs = interfaces[index][:, 0] / 1000.0
    zs = interfaces[index][:, 1] / 1000.0 - length_z
    return lambda x: np.interp(x, xs, zs)

### 2.3 Source and receiver placement on the interface

`acquisition_on_interface` positions the source and receivers relative to a non-planar interface $z(x)$, ensuring they stay at a fixed height above/below it regardless of the interface shape. In this notebook, it is used to place the source and receivers relative to the sinusoidal interface.

In [ ]:
def acquisition_on_interface(z_int, x_source, height_source, offset, distance):
    z_at = lambda x: float(z_int(np.array([x]))[0])
    return {
        "source_locations": [(z_at(x_source) + height_source, x_source)],
        "receiver_location": [(z_at(x_source - offset) + distance, x_source - offset)],
        "solid_receiver_locations": [(z_at(x_source + offset) - distance, x_source + offset)],
    }

### 2.4 Plotting functions

Two plotting utilities:
- `plot_mesh_xz`: plots one or more Firedrake meshes with swapped axes (X horizontal, Z vertical, increasing downward), which is the standard seismic convention.
- `plot_model_xz`: plots the velocity model on the sub-meshes (fluid and solid separately) with source and receiver positions overlaid.

> **Note:** Additional visualization functions will be included in a future version of this notebook.

In [ ]:
def plot_mesh_xz(meshes, title, filename, length_x, length_z):
    fig, ax = plt.subplots(figsize=(8, 8))
    swap = Affine2D().from_values(0, 1, 1, 0, 0, 0) + ax.transData
    for mesh, color in meshes:
        kw = {"edgecolors": color} if color else {}
        for coll in triplot(mesh, axes=ax, interior_kw=kw):
            coll.set_transform(swap)
    ax.set_xlim(0.0, length_x)
    ax.set_ylim(-length_z, 0.0)
    ax.set_aspect("equal")
    ax.set_xlabel("X (km)", fontsize=18)
    ax.set_ylabel("Z (km)", fontsize=18)
    ax.set_title(title, fontsize=22)
    ax.tick_params(axis="both", labelsize=18)
    fig.tight_layout()
    fig.savefig(filename, dpi=100)
    plt.close(fig)


def plot_model_xz(wave, filename, dictionary, length_x, length_z):
    vf = Function(FunctionSpace(wave.submesh_fluid, "CG", 1)).interpolate(
        velocity_fluid(wave))
    vs = Function(FunctionSpace(wave.submesh_solid, "CG", 1)).interpolate(
        ((wave.lmbda + 2 * wave.mu) / wave.rho) ** 0.5)
    vmin = min(vf.dat.data_ro.min(), vs.dat.data_ro.min())
    vmax = max(vf.dat.data_ro.max(), vs.dat.data_ro.max())

    fig, ax = plt.subplots(figsize=(9, 9))
    swap = Affine2D().from_values(0, 1, 1, 0, 0, 0) + ax.transData
    for f in (vf, vs):
        img = tripcolor(f, axes=ax, vmin=vmin, vmax=vmax)
        img.set_transform(swap)

    acq = dictionary["acquisition"]
    for key, marker, color, label in (
        ("source_locations", "*", "green", "Source"),
        ("receiver_locations", "v", "red", "Receiver (fluid)"),
        ("solid_receiver_locations", "^", "orange", "Receiver (solid)"),
    ):
        pts = np.asarray(acq[key])
        ax.scatter(pts[:, 1], pts[:, 0], marker=marker, c=color, s=120, label=label, zorder=5)

    ax.set_xlim(0.0, length_x)
    ax.set_ylim(-length_z, 0.0)
    ax.set_aspect("equal")
    ax.set_xlabel("X (km)", fontsize=18)
    ax.set_ylabel("Z (km)", fontsize=18)
    ax.tick_params(axis="both", labelsize=18)
    ax.legend(loc="best", fontsize=14)
    cbar = fig.colorbar(img, ax=ax, orientation="horizontal")
    cbar.set_label("Velocity (km/s)", fontsize=18)
    cbar.ax.tick_params(labelsize=18)
    fig.tight_layout()
    fig.savefig(filename, dpi=100)
    plt.close(fig)


def plot_snapshot_xz(wave, filename, length_x, length_z):
    """Plot pressure (fluid) and sigma_zz (solid) together."""
    fig, ax = plt.subplots(figsize=(9, 9))
    swap = Affine2D().from_values(0, 1, 1, 0, 0, 0) + ax.transData

    p_fluid = wave.X_n.sub(0)
    sigma_solid = wave._compute_sigma_zz()

    vmax = max(
        np.abs(p_fluid.dat.data_ro).max(),
        np.abs(sigma_solid.dat.data_ro).max()
    )
    if vmax == 0:
        vmax = 1.0

    for field in (p_fluid, sigma_solid):
        img = tripcolor(field, axes=ax, vmin=-vmax, vmax=vmax, cmap="seismic")
        img.set_transform(swap)

    ax.set_xlim(0.0, length_x)
    ax.set_ylim(-length_z, 0.0)
    ax.set_aspect("equal")
    ax.set_xlabel("X (km)", fontsize=18)
    ax.set_ylabel("Z (km)", fontsize=18)
    ax.set_title("Final snapshot: pressure (fluid) and $\\sigma_{zz}$ (solid)", fontsize=20)
    ax.tick_params(axis="both", labelsize=18)
    cbar = fig.colorbar(img, ax=ax, orientation="horizontal")
    cbar.set_label("Amplitude", fontsize=18)
    cbar.ax.tick_params(labelsize=18)
    fig.tight_layout()
    fig.savefig(filename, dpi=100)
    plt.close(fig)


def plot_receivers(t_p, p_spyro, t_u, u_solid, filename, figsize=(12, 8)):
    """Plot pressure at fluid receiver and displacement at solid receiver."""
    fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=False)

    axes[0].plot(t_p, p_spyro, color="tab:blue", lw=1.5)
    axes[0].set_title("Fluid receiver — pressure", fontsize=16)
    axes[0].set_ylabel("Amplitude", fontsize=14)
    axes[0].set_xlabel("Time (s)", fontsize=14)
    axes[0].tick_params(labelsize=13)
    axes[0].grid(True, alpha=0.3)

    labels = ["uz (vertical)", "ux (horizontal)"]
    colors = ["tab:orange", "tab:green"]
    for i, (label, color) in enumerate(zip(labels, colors)):
        axes[i + 1].plot(t_u, u_solid[:, i], color=color, lw=1.5)
        axes[i + 1].set_title(f"Solid receiver — displacement {label}", fontsize=16)
        axes[i + 1].set_ylabel("Amplitude", fontsize=14)
        axes[i + 1].set_xlabel("Time (s)", fontsize=14)
        axes[i + 1].tick_params(labelsize=13)
        axes[i + 1].grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(filename, dpi=100)
    plt.close(fig)

## 3. Simulation settings

Three top-level switches control the run:
- `USE_MONOLITHIC`: if `True`, solves the coupled fluid and solid systems simultaneously in a single block system; if `False`, uses the sequential scheme. **In this example, the sequential scheme is used.**
- `MESH_TYPE`: `"firedrake_mesh"` uses Firedrake's built-in mesh generator; `"gmsh_mesh"` uses an external mesh built from a velocity model with a non-planar interface. **In this example, `"gmsh_mesh"` is used.**
- `OUT`: directory where all output files will be saved.

In [ ]:
USE_MONOLITHIC = False
MESH_TYPE = "gmsh_mesh"
OUT = "results"
os.makedirs(OUT, exist_ok=True)

## 4. Case definition

This notebook runs the **small 2D case with a sinusoidal non-planar interface** (`case_small_2D_irregular`). The domain is $1 \times 1$ km, with a sinusoidal fluid–solid interface of amplitude $0.02$ km and wavelength $0.25$ km centered at $z = -0.5$ km. Fourth-order quadrilateral elements (Q4) are used with a mesh generated by GMSH.

The material parameters are:
- $V_\mathrm{fluid} = 1.5$ km/s, $\rho_\mathrm{fluid} = 1.0$ g/cm³
- $V_{P,\mathrm{solid}} = 2.0$ km/s, $V_{S,\mathrm{solid}} = 1.2$ km/s, $\rho_\mathrm{solid} = 2.0$ g/cm³
- Dominant frequency: $25$ Hz, final time: $0.22$ s

In [ ]:
case_small_2D = {
    "cell_type": "Q",
    "degree": 4,
    "length_z": 1.0,
    "length_x": 1.0,
    "length_y": 0.0,
    "edge_length": 0.010,
    "interface_z": -0.5,
    "source_locations": [(-0.45, 0.5)],
    "frequency": 25,
    "receiver_location": [(-0.49, 0.49)],
    "solid_receiver_locations": [(-0.51, 0.51)],
    "final_time": 0.22,
    "dt": 0.0001,
    "velocity_fluid": None,
    "bulk_modulus": 2.25,
    "density_fluid": 1.0,
    "density_solid": 2.0,
    "p_wave_velocity": 2.0,
    "s_wave_velocity": 1.2,
    "segy_dz": 0.002,
    "segy_interface": -0.5,
    "cells_per_wavelength": 6.0,
}

# Sinusoidal non-planar interface: z_int(x) = -0.5 + 0.02*sin(2*pi*x / 0.25)
small_irregular_interface = lambda x: -0.5 + 0.02 * np.sin(2 * np.pi * x / 0.25)

selected_case = dict(
    case_small_2D,
    segy_interface=small_irregular_interface,
    segy_dz=0.0005,
    segy_dx=0.010,
)

### 4.1 Run configuration

The variables below define the case name, mesh type, dimension, and solution scheme. The fluid P-wave velocity is derived from the bulk modulus and fluid density.

In [ ]:
CASE_NAME = "case_small_2d_irregular_topography"
DIMENSION = 2
SCHEME = "monolithic" if USE_MONOLITHIC else "sequential"
TAG = f"{CASE_NAME}_{SCHEME}_{DIMENSION}d"

VP_WATER = np.sqrt(selected_case["bulk_modulus"] / selected_case["density_fluid"])

print(f"Running: {TAG}")
print(f"Mesh type: {MESH_TYPE} | Dimension: {DIMENSION}D | Scheme: {SCHEME}")
print(f"Fluid P-wave velocity: {VP_WATER:.3f} km/s")

## 5. Simulation dictionary

The `dictionary` is the main input structure for `AcousticElasticWave`. It groups all parameters into sections:
- **`options`**: element type, polynomial degree, and dimension.
- **`parallelism`**: MPI parallelization strategy.
- **`mesh`**: domain dimensions, mesh type, and meshing parameters for the GMSH-generated mesh.
- **`acquisition`**: source wavelet type, source and receiver positions.
- **`time_axis`**: time stepping parameters.
- **`visualization`**: output files for Paraview (PVD) and snapshots.
- **`synthetic_data`**: material properties (bulk modulus, densities, and wave velocities).
- **`absorbing_boundary_conditions`**: ABC activation.

In [ ]:
dictionary = {}

dictionary["options"] = {
    "cell_type": selected_case["cell_type"],
    "variant": "lumped",
    "degree": selected_case["degree"],
    "dimension": DIMENSION,
}

dictionary["parallelism"] = {
    "type": "automatic",
}

dictionary["mesh"] = {
    "length_z": selected_case["length_z"],
    "length_x": selected_case["length_x"],
    "length_y": selected_case["length_y"],
    "mesh_file": None,
    "mesh_type": MESH_TYPE,
    "edge_length": selected_case["edge_length"],
    "interface_z": selected_case["interface_z"],
}

segy_file = f"{OUT}/model_{TAG}.segy"
build_segy(
    segy_file,
    selected_case["length_z"],
    selected_case["length_x"],
    selected_case["segy_dz"],
    selected_case["segy_interface"],
    VP_WATER, selected_case["s_wave_velocity"],
    dx=selected_case.get("segy_dx")
)
dictionary["mesh"].pop("edge_length")
dictionary["mesh"].update({
    "output_filename": f"{OUT}/mesh_{TAG}.msh",
    "segy_velocity_model": segy_file,
    "cells_per_wavelength": selected_case["cells_per_wavelength"],
    "frequency": selected_case["frequency"],
    "min_element_size": selected_case["edge_length"],
    "grade": 0.05,
    "structured_mesh": selected_case["cell_type"] == "Q",
    "apply_winslow": False,
    "winslow_iterations": 2000,
    "vp_water": VP_WATER,
    "water_interface": True,
    "water_search_value": VP_WATER,
})

dictionary["acquisition"] = {
    "source_type": "ricker",
    "source_locations": selected_case["source_locations"],
    "frequency": selected_case["frequency"],
    "delay": 1.0 / selected_case["frequency"],
    "delay_type": "time",
    "amplitude": 1.0,
    "receiver_locations": selected_case["receiver_location"],
    "solid_receiver_locations": selected_case["solid_receiver_locations"],
    "use_vertex_only_mesh": False,
}

dictionary["time_axis"] = {
    "initial_time": 0.0,
    "final_time": selected_case["final_time"],
    "dt": selected_case["dt"],
    "output_frequency": int(round(0.005 / selected_case["dt"])),
    "gradient_sampling_frequency": 1,
}

dictionary["visualization"] = {
    "forward_output": True,
    "forward_output_filename": f"{OUT}/pressure_{TAG}.pvd",
    "fwi_velocity_model_output": False,
    "velocity_model_filename": None,
    "gradient_output": False,
    "gradient_filename": None,
    "debug_output": False,
    "displacement_output": False,
    "displacement_output_filename": f"{OUT}/displacement_{TAG}.pvd",
    "snapshot_frequency": 20,
    "snapshot_output_dir": f"{OUT}/snapshots_{TAG}",
    "sigma_zz_output": False,
}

dictionary["synthetic_data"] = {
    "type": "object",
    "bulk_modulus": selected_case["bulk_modulus"],
    "density_fluid": selected_case["density_fluid"],
    "density_solid": selected_case["density_solid"],
    "p_wave_velocity": selected_case["p_wave_velocity"],
    "s_wave_velocity": selected_case["s_wave_velocity"],
    "real_velocity_file": None,
}

dictionary["absorbing_boundary_conditions"] = {
    "status": False,
}

## 6. Solver initialization

`AcousticElasticWave` reads the dictionary, builds the mesh, creates the fluid and solid sub-meshes via Firedrake's `Submesh`, and sets up the function spaces and variational forms. The diagnostic prints below verify that the sub-mesh boundary markers are correctly assigned and that the elements are on the correct side of the sinusoidal interface.

In [ ]:
Wave_obj = AcousticElasticWave(dictionary=dictionary)

# Check boundary markers on each sub-mesh
print("Fluid exterior facet markers:",
      Wave_obj.scalar_function_space.mesh().exterior_facets.unique_markers)
print("Solid exterior facet markers:",
      Wave_obj.vector_function_space.mesh().exterior_facets.unique_markers)

# Verify that elements are on the correct side of the sinusoidal interface
for name, sub in (("fluid", Wave_obj.submesh_fluid), ("solid", Wave_obj.submesh_solid)):
    V = fire.VectorFunctionSpace(sub, "DG", 0)
    c = fire.Function(V).interpolate(fire.SpatialCoordinate(sub)).dat.data_ro
    zi = -0.5 + 0.02 * np.sin(2 * np.pi * c[:, 1] / 0.25)
    wrong = (c[:, 0] < zi) if name == "fluid" else (c[:, 0] > zi)
    print(f"{name} - elements on wrong side: {wrong.sum()} of {len(c)}")

# Check interface length (sinusoidal exact: ~1.06 km)
L = fire.assemble(fire.Constant(1.0) * Wave_obj.ds_int(Wave_obj.interface_id))
print(f"Interface length: {L:.4f} km  (sinusoidal exact: ~1.06 km)")

Wave_obj.use_monolithic = USE_MONOLITHIC

LX = selected_case["length_x"]
LZ = selected_case["length_z"]

### 6.1 Mesh visualization

The plots below show the parent mesh and the child sub-meshes (fluid in blue, solid in orange).

In [ ]:
parent_file = f"{OUT}/mesh_parent_{TAG}.png"
child_file  = f"{OUT}/mesh_child_{TAG}.png"

plot_mesh_xz([(Wave_obj.mesh, None)], "Parent mesh", parent_file, LX, LZ)
plot_mesh_xz(
    [(Wave_obj.submesh_fluid, "blue"), (Wave_obj.submesh_solid, "orange")],
    "Child submeshes", child_file, LX, LZ
)

display(Image(parent_file, width=500))
display(Image(child_file,  width=500))

## 7. Forward solve

Runs the time-stepping loop. Elapsed time and peak memory usage are measured and printed at the end.

In [ ]:
t_start = time.perf_counter()
Wave_obj.forward_solve()
elapsed = time.perf_counter() - t_start
mem_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0

print(f"Case: {TAG}")
print(f"Elapsed time: {elapsed:.2f} s")
print(f"Peak memory:  {mem_mb:.2f} MB")

### 7.1 Velocity model and wavefield snapshot

The velocity model plot shows the fluid and solid domains with source and receiver positions. The snapshot shows the pressure field at the final time step.

In [ ]:
model_file    = f"{OUT}/model_{TAG}.png"
snapshot_file = f"{OUT}/snapshot_{TAG}.png"

plot_model_xz(Wave_obj, model_file, dictionary, LX, LZ)
plot_snapshot_xz(Wave_obj, snapshot_file, LX, LZ)

display(Image(model_file,    width=500))
display(Image(snapshot_file, width=500))

## 8. Save results

Receiver time series are saved to a `.npz` file containing:
- `p_spyro`: pressure recorded at the fluid receiver.
- `u_solid`: displacement components recorded at the solid receiver.
- `dt` and `final_time`: time axis parameters needed for post-processing.

In a parallel run, `communicate` gathers the solid receiver data from all MPI processes before saving. Only rank 0 writes to disk.

In [ ]:
p_spyro = np.asarray(Wave_obj.forward_solution_receivers)[:, 0]
u_all   = communicate(np.asarray(Wave_obj.solid_receiver_history), Wave_obj.comm)
u_solid = u_all[:, 0, :]

n_procs = Wave_obj.comm.comm.size
rank    = Wave_obj.comm.comm.rank

if rank == 0:
    np.savez(
        f"{OUT}/spyro_receiver_data_{TAG}.npz",
        dt=dictionary["time_axis"]["dt"],
        final_time=dictionary["time_axis"]["final_time"],
        p_spyro=p_spyro,
        u_solid=u_solid,
    )
    print(f"[OK] Saved: {OUT}/spyro_receiver_data_{TAG}.npz")

## 9. Receiver seismograms

Time series recorded at the fluid receiver (pressure) and solid receiver (displacement components uz and ux).

In [ ]:
dt_case = dictionary["time_axis"]["dt"]
t_p = np.arange(len(p_spyro)) * dt_case
t_u = np.arange(len(u_solid)) * dt_case

recv_file = f"{OUT}/receivers_{TAG}.png"
plot_receivers(t_p, p_spyro, t_u, u_solid, recv_file)
display(Image(recv_file, width=700))